# Semantic Correspondence — Report di Analisi Completo

Questo notebook analizza tutti i risultati prodotti dal pipeline su Google Colab.  
Gira **completamente in locale su Mac**, senza GPU.

## Setup
1. Scarica da Google Drive la cartella:  
   `MyDrive/Colab Notebooks/AML_results/runs/pipeline_exports/`
2. Mettila accanto a questo notebook (oppure aggiorna `EXPORTS_DIR` sotto).
3. Installa le dipendenze (una volta sola):
   ```bash
   pip install pandas matplotlib seaborn numpy
   ```
4. Esegui tutte le celle: **Cell → Run All**.

I grafici vengono salvati automaticamente in `./figures/`.

---
**Sezioni del report**
1. Caricamento dati · 2. Tabella aggregata · 3. Progressione per stage  
4. Confronto backbone · 5. Sweep profondità FT · 6. Impatto WSA  
7. LoRA vs Fine-tuning · 8. PCK per categoria · 9. Analisi difficoltà · 10. Riepilogo

In [ ]:
from pathlib import Path

# ── Cambia questo percorso con la cartella scaricata da Drive ────────────────
EXPORTS_DIR = Path("./pipeline_exports")
# ────────────────────────────────────────────────────────────────────────────

FIGURES_DIR = Path("./figures")
FIGURES_DIR.mkdir(exist_ok=True)

print(f"EXPORTS_DIR : {EXPORTS_DIR.resolve()}")
print(f"Esiste      : {EXPORTS_DIR.exists()}")
if EXPORTS_DIR.exists():
    print(f"File trovati: {[f.name for f in sorted(EXPORTS_DIR.iterdir())]}")

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
sns.set_theme(style="whitegrid", palette="muted")
print("Librerie caricate.")

## 1 · Caricamento dati

In [ ]:
def load_json(path: Path):
    if not path.exists():
        print(f"[SKIP] File non trovato: {path.name}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"[OK]   {path.name}  ({len(data)} voci)")
    return data

results          = load_json(EXPORTS_DIR / "pck_results.json") or []
per_category_raw = load_json(EXPORTS_DIR / "pck_results_per_category.json") or []
difficulty_raw   = load_json(EXPORTS_DIR / "pck_results_by_difficulty_flag.json") or []
per_image_raw    = load_json(EXPORTS_DIR / "pck_results_per_image.json") or []

print(f"\nTotale run di valutazione: {len(results)}")

In [ ]:
# ── Costanti globali e parsing nomi run ──────────────────────────────────────
BACKBONES = ["dinov2_vitb14", "dinov3_vitb16", "sam_vit_b"]
BB_LABEL  = {
    "dinov2_vitb14": "DINOv2 ViT-B/14",
    "dinov3_vitb16": "DINOv3 ViT-B/16",
    "sam_vit_b":     "SAM ViT-B",
}
BB_COLOR = {
    "dinov2_vitb14": "#2196F3",
    "dinov3_vitb16": "#FF9800",
    "sam_vit_b":     "#4CAF50",
}
SPAIR_CATS = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car",
    "cat", "chair", "cow", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "train", "tvmonitor",
]
FLAGS = ["viewpoint_variation", "scale_variation", "truncation", "occlusion"]
FLAG_LABELS = {
    "viewpoint_variation": "Variazione\nViewpoint",
    "scale_variation":     "Variazione\nScale",
    "truncation":          "Truncation",
    "occlusion":           "Occlusion",
}

def parse_run(name: str) -> dict:
    """Decompone il nome run: backbone, stage, blocks, wsa."""
    for bb in BACKBONES:
        if name == bb or name.startswith(bb + "_"):
            rest = name[len(bb):].lstrip("_")
            wsa  = rest.endswith("_wsa") or rest == "wsa"
            if wsa:
                rest = rest[:-4].rstrip("_") if rest != "wsa" else ""
            if rest in ("", "baseline"):
                return dict(backbone=bb, stage="baseline", blocks=None, wsa=wsa)
            if rest.startswith("ft_lb"):
                try:
                    return dict(backbone=bb, stage="finetune", blocks=int(rest[5:]), wsa=wsa)
                except ValueError:
                    pass
            if rest == "lora":
                return dict(backbone=bb, stage="lora", blocks=None, wsa=wsa)
    return dict(backbone=name, stage="unknown", blocks=None, wsa=False)

def stage_label(stage, blocks=None):
    return {"baseline": "Baseline",
            "finetune": f"Fine-tuning (lb={int(blocks)})" if blocks else "Fine-tuning",
            "lora":     "LoRA"}.get(stage, stage)

def run_label(name):
    p = parse_run(name)
    bb  = BB_LABEL.get(p["backbone"], p["backbone"])
    s   = stage_label(p["stage"], p["blocks"])
    wsa = " + WSA" if p["wsa"] else ""
    return f"{bb} | {s}{wsa}"

# ── DataFrame principale ──────────────────────────────────────────────────────
rows = []
for r in results:
    p = parse_run(r["name"])
    m = r.get("metrics", {})
    rows.append({
        "name":     r["name"],
        "backbone": p["backbone"],
        "stage":    p["stage"],
        "blocks":   p["blocks"],
        "wsa":      p["wsa"],
        "pck@0.05": m.get("pck@0.05", float("nan")),
        "pck@0.1":  m.get("pck@0.1",  float("nan")),
        "pck@0.2":  m.get("pck@0.2",  float("nan")),
    })

df = pd.DataFrame(rows)
backbones_present = [b for b in BACKBONES if b in df["backbone"].values]
STAGE_SORT = {"baseline": 0, "finetune": 1, "lora": 2, "unknown": 9}

print(f"DataFrame: {df.shape[0]} run × {df.shape[1]} colonne")
print(f"Backbone presenti: {backbones_present}")
print("\nRun trovati:")
for n in df["name"].tolist():
    print(f"  {n}")

## 2 · Tabella aggregata completa
PCK@{0.05, 0.1, 0.2} per tutti i run. Il colore scala da bianco (peggiore) a verde (migliore) colonna per colonna.

In [ ]:
df_sorted = df.copy()
df_sorted["_sbb"]  = df_sorted["backbone"].map(lambda b: BACKBONES.index(b) if b in BACKBONES else 9)
df_sorted["_ss"]   = df_sorted["stage"].map(STAGE_SORT)
df_sorted["_sblk"] = df_sorted["blocks"].fillna(0)
df_sorted["_swsa"] = df_sorted["wsa"].astype(int)
df_sorted = df_sorted.sort_values(["_sbb", "_ss", "_sblk", "_swsa"])
df_sorted["Metodo"] = df_sorted["name"].map(run_label)

table = df_sorted[["Metodo", "pck@0.05", "pck@0.1", "pck@0.2"]].set_index("Metodo")
table.columns = ["PCK@0.05", "PCK@0.10", "PCK@0.20"]

styled = (
    table.style
    .format("{:.4f}")
    .background_gradient(cmap="YlGn", axis=0)
    .set_caption("PCK@α — tutti i metodi (↑ migliore)")
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "14px"), ("font-weight", "bold")]}])
)
display(styled)

table.to_csv(FIGURES_DIR / "pck_table_full.csv")
print("Salvato: figures/pck_table_full.csv")

## 3 · Progressione per stage
Per ogni backbone: come migliora il PCK@0.1 passando da Baseline → Best Fine-tuning → LoRA (con e senza WSA).

In [ ]:
METRIC = "pck@0.1"

stage_steps = [
    ("baseline", False, "Baseline"),
    ("baseline", True,  "Baseline\n+WSA"),
    ("finetune", False, "Best FT"),
    ("finetune", True,  "Best FT\n+WSA"),
    ("lora",     False, "LoRA"),
    ("lora",     True,  "LoRA\n+WSA"),
]

fig, axes = plt.subplots(1, len(backbones_present),
                         figsize=(5 * len(backbones_present), 5), sharey=True)
if len(backbones_present) == 1:
    axes = [axes]

for ax, bb in zip(axes, backbones_present):
    sub = df[df["backbone"] == bb]
    vals, xlbls = [], []
    for stage, wsa, lbl in stage_steps:
        s2 = sub[(sub["stage"] == stage) & (sub["wsa"] == wsa)]
        if s2.empty or s2[METRIC].isna().all():
            continue
        vals.append(s2[METRIC].max())  # miglior variante (es. best blocks)
        xlbls.append(lbl)

    x = np.arange(len(vals))
    alphas_bar = [0.95 if i % 2 == 0 else 0.5 for i in range(len(vals))]
    bars = ax.bar(x, vals, color=BB_COLOR[bb], edgecolor="white", linewidth=1.2,
                  alpha=1.0)  # usiamo facecolor per simulare alpha variabile
    for i, (bar, v, ab) in enumerate(zip(bars, vals, alphas_bar)):
        bar.set_alpha(ab)
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(xlbls, fontsize=9)
    ax.set_title(BB_LABEL[bb], fontweight="bold")
    ax.set_xlabel("Stage")
    ax.set_ylabel(METRIC.upper())
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    # Nota: barre con alpha ridotto = variante +WSA
    ax.text(0.97, 0.03, "chiaro = +WSA", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8, color="gray")

fig.suptitle("Progressione PCK@0.1 per stage — tutte le backbone",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage_progression.png", bbox_inches="tight")
plt.show()
print("Salvato: figures/stage_progression.png")

## 4 · Confronto backbone — miglior metodo per ciascuna
Per ogni backbone viene mostrato il run con PCK@0.1 più alto a tutti e tre i threshold α.

In [ ]:
alphas = ["pck@0.05", "pck@0.1", "pck@0.2"]
alpha_labels = ["PCK@0.05", "PCK@0.10", "PCK@0.20"]

best_per_bb = {}
for bb in backbones_present:
    sub = df[df["backbone"] == bb]
    if sub.empty:
        continue
    best_per_bb[bb] = sub.loc[sub["pck@0.1"].idxmax()]

x = np.arange(len(alphas))
width = 0.7 / max(len(best_per_bb), 1)
fig, ax = plt.subplots(figsize=(9, 5))

for i, (bb, row) in enumerate(best_per_bb.items()):
    vals   = [row[a] for a in alphas]
    offset = (i - len(best_per_bb) / 2 + 0.5) * width
    bars   = ax.bar(x + offset, vals, width * 0.9,
                    label=f"{BB_LABEL[bb]}\n({row['name']})",
                    color=BB_COLOR[bb], edgecolor="white", linewidth=1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(alpha_labels, fontsize=11)
ax.set_ylabel("PCK@α")
ax.set_title("Migliore configurazione per backbone (tutti i threshold α)", fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "backbone_comparison.png", bbox_inches="tight")
plt.show()
print("Salvato: figures/backbone_comparison.png")

## 5 · Sweep profondità fine-tuning
`last_blocks ∈ {1, 2, 4}` — quanti blocchi transformer sbloccare durante il fine-tuning.

In [ ]:
ft_df = df[df["stage"] == "finetune"].copy()

if ft_df.empty:
    print("Nessun run di fine-tuning trovato.")
else:
    fig, axes = plt.subplots(1, len(backbones_present),
                             figsize=(5 * len(backbones_present), 5), sharey=True)
    if len(backbones_present) == 1:
        axes = [axes]

    for ax, bb in zip(axes, backbones_present):
        sub = ft_df[ft_df["backbone"] == bb].sort_values("blocks")
        if sub.empty:
            ax.set_title(BB_LABEL[bb] + "\n(nessun dato)")
            continue

        blocks_vals = sorted(sub["blocks"].dropna().unique())
        x = np.arange(len(blocks_vals))
        w = 0.35

        for offset, wsa, alpha_bar, lbl in [
            (-w / 2, False, 0.95, "senza WSA"),
            ( w / 2, True,  0.50, "con WSA"),
        ]:
            s2 = sub[sub["wsa"] == wsa]
            vals_per_blk = [
                s2[s2["blocks"] == b]["pck@0.1"].values[0]
                if not s2[s2["blocks"] == b].empty else float("nan")
                for b in blocks_vals
            ]
            bars = ax.bar(x + offset, vals_per_blk, w,
                          label=lbl, color=BB_COLOR[bb],
                          alpha=alpha_bar, edgecolor="white")
            for bar, v in zip(bars, vals_per_blk):
                if not np.isnan(v):
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.002,
                            f"{v:.3f}", ha="center", va="bottom", fontsize=8.5)

        ax.set_xticks(x)
        ax.set_xticklabels([f"lb={int(b)}" for b in blocks_vals])
        ax.set_title(BB_LABEL[bb], fontweight="bold")
        ax.set_ylabel("PCK@0.1")
        ax.legend(fontsize=9)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

    fig.suptitle("Impatto del numero di blocchi sbloccati (Fine-tuning depth sweep)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "finetune_depth_sweep.png", bbox_inches="tight")
    plt.show()
    print("Salvato: figures/finetune_depth_sweep.png")

## 6 · Impatto Window Soft-Argmax (WSA)
Δ PCK = (con WSA) − (senza WSA) per ogni run. Valori positivi = WSA aiuta.

In [ ]:
no_wsa  = df[df["wsa"] == False].copy()
yes_wsa = df[df["wsa"] == True].copy()

merged = no_wsa.merge(
    yes_wsa[["backbone", "stage", "blocks", "pck@0.05", "pck@0.1", "pck@0.2"]],
    on=["backbone", "stage", "blocks"],
    suffixes=("_base", "_wsa"),
)

if merged.empty:
    print("Nessuna coppia (base, +WSA) trovata.")
else:
    for a in ["pck@0.05", "pck@0.1", "pck@0.2"]:
        merged[f"delta_{a}"] = merged[f"{a}_wsa"] - merged[f"{a}_base"]

    merged_s = merged.sort_values(["backbone", "stage", "blocks"])

    def short_label(row):
        s = {"baseline": "Base",
             "finetune": f"FT lb={int(row['blocks'])}" if row["blocks"] else "FT",
             "lora": "LoRA"}.get(row["stage"], row["stage"])
        bb = BB_LABEL.get(row["backbone"], row["backbone"]).replace(" ", "\n")
        return f"{bb}\n{s}"

    xlabels = [short_label(r) for _, r in merged_s.iterrows()]
    colors_b = [BB_COLOR.get(r["backbone"], "gray") for _, r in merged_s.iterrows()]
    x = np.arange(len(merged_s))
    w = 0.22

    fig, ax = plt.subplots(figsize=(max(10, len(merged_s) * 1.1), 5))
    for offset, a, alpha_v, lbl in [
        (-w, "pck@0.05", 0.5, "Δ PCK@0.05"),
        ( 0, "pck@0.1",  0.9, "Δ PCK@0.10"),
        ( w, "pck@0.2",  0.6, "Δ PCK@0.20"),
    ]:
        deltas = merged_s[f"delta_{a}"].values
        ax.bar(x + offset, deltas, w, label=lbl,
               color=colors_b, alpha=alpha_v, edgecolor="white")

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=8.5)
    ax.set_ylabel("Δ PCK  (WSA − base)")
    ax.set_title("Impatto WSA su ogni metodo", fontweight="bold")

    bb_patches = [mpatches.Patch(color=BB_COLOR[b], label=BB_LABEL[b])
                  for b in backbones_present]
    alpha_handles, alpha_lbls = ax.get_legend_handles_labels()
    ax.legend(handles=bb_patches + alpha_handles[:3],
              labels=[BB_LABEL[b] for b in backbones_present] + alpha_lbls[:3],
              fontsize=9, ncol=2, loc="upper left")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "wsa_impact.png", bbox_inches="tight")
    plt.show()

    print("\nDelta medio WSA per stage (PCK@0.1):")
    print(merged_s.groupby(["backbone", "stage"])["delta_pck@0.1"].mean().round(4))
    print("Salvato: figures/wsa_impact.png")

## 7 · LoRA vs Fine-tuning
Confronto diretto tra Baseline, miglior Fine-tuning e LoRA per ogni backbone, con e senza WSA.

In [ ]:
groups = [
    ("baseline", False, "Baseline"),
    ("baseline", True,  "Baseline\n+WSA"),
    ("finetune", False, "Best FT"),
    ("finetune", True,  "Best FT\n+WSA"),
    ("lora",     False, "LoRA"),
    ("lora",     True,  "LoRA+WSA"),
]
METRIC_PLOT = "pck@0.1"

x     = np.arange(len(groups))
n_bb  = len(backbones_present)
width = 0.75 / max(n_bb, 1)

fig, ax = plt.subplots(figsize=(12, 5))
for i, bb in enumerate(backbones_present):
    sub = df[df["backbone"] == bb]
    vals = []
    for stage, wsa, _ in groups:
        s2 = sub[(sub["stage"] == stage) & (sub["wsa"] == wsa)]
        vals.append(s2[METRIC_PLOT].max() if not s2.empty else float("nan"))

    offset = (i - n_bb / 2 + 0.5) * width
    bar_alphas = [1.0 if j % 2 == 0 else 0.55 for j in range(len(vals))]
    bars = ax.bar(x + offset, vals, width * 0.92,
                  color=BB_COLOR[bb], label=BB_LABEL[bb], edgecolor="white")
    for bar, v, ab in zip(bars, vals, bar_alphas):
        bar.set_alpha(ab)
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.001,
                    f"{v:.3f}", ha="center", va="bottom",
                    fontsize=7.5, rotation=40)

ax.set_xticks(x)
ax.set_xticklabels([lbl for _, _, lbl in groups], fontsize=10)
ax.set_ylabel(METRIC_PLOT.upper())
ax.set_title("Confronto metodi per backbone — PCK@0.1  (chiaro = +WSA)",
             fontweight="bold")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lora_vs_finetune.png", bbox_inches="tight")
plt.show()
print("Salvato: figures/lora_vs_finetune.png")

## 8 · PCK per categoria (heatmap)
Ogni riga è un metodo, ogni colonna una delle 18 categorie SPair-71k, il valore è PCK@0.1.  
Vengono mostrati solo i run senza WSA per ridurre la ridondanza.

In [ ]:
if not per_category_raw:
    print("File pck_results_per_category.json non trovato — salta sezione.")
else:
    ALPHA_CAT = "pck@0.1"

    # Costruisce matrice (run × categoria)
    cat_records = {}
    for entry in per_category_raw:
        row_d = {}
        for cat in SPAIR_CATS:
            row_d[cat] = entry["categories"].get(cat, {}).get(ALPHA_CAT, float("nan"))
        cat_records[entry["name"]] = row_d

    cat_df = pd.DataFrame(cat_records).T   # run × categoria

    # Ordina per backbone e stage, mostra solo run senza WSA
    no_wsa_names = df_sorted[~df_sorted["wsa"]]["name"].tolist()
    cat_df = cat_df.loc[[n for n in no_wsa_names if n in cat_df.index]]

    if cat_df.empty:
        # Fallback: mostra tutto
        cat_df = pd.DataFrame(cat_records).T

    cat_df.index = [run_label(n) for n in cat_df.index]

    fig, ax = plt.subplots(figsize=(max(14, len(SPAIR_CATS) * 0.82),
                                    max(6, len(cat_df) * 0.5)))
    sns.heatmap(
        cat_df.astype(float),
        ax=ax,
        cmap="YlOrRd",
        annot=True,
        fmt=".2f",
        annot_kws={"size": 7.5},
        linewidths=0.4,
        cbar_kws={"label": "PCK@0.1"},
        vmin=0.0,
    )
    ax.set_title("PCK@0.1 per categoria SPair-71k",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Categoria")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=40, labelsize=9)
    ax.tick_params(axis="y", labelsize=8.5)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "pck_per_category_heatmap.png", bbox_inches="tight")
    plt.show()
    print("Salvato: figures/pck_per_category_heatmap.png")

    print("\nRanking categorie per PCK@0.1 medio (tra tutti i metodi):")
    print(cat_df.mean().sort_values(ascending=False).round(4).to_string())

## 9 · Analisi per flag di difficoltà
SPair-71k etichetta ogni coppia con 4 flag binari.  
Confronto PCK@0.1 quando il flag è **assente (0)** vs **presente (1)** — per il miglior run di ogni backbone.

In [ ]:
if not difficulty_raw:
    print("File pck_results_by_difficulty_flag.json non trovato — salta sezione.")
else:
    # Miglior run per backbone (basato su PCK@0.1)
    best_names = {}
    for bb in backbones_present:
        sub = df[df["backbone"] == bb]
        if not sub.empty:
            best_names[bb] = sub.loc[sub["pck@0.1"].idxmax(), "name"]

    diff_by_name = {r["name"]: r for r in difficulty_raw}
    runs_diff = {bb: diff_by_name[n]
                 for bb, n in best_names.items() if n in diff_by_name}

    if not runs_diff:
        print("Nessun run compatibile con i dati di difficoltà.")
    else:
        fig, axes = plt.subplots(1, len(FLAGS),
                                 figsize=(4.5 * len(FLAGS), 5), sharey=False)

        for ax, flag in zip(axes, FLAGS):
            data_rows = []
            for bb, entry in runs_diff.items():
                flag_data = entry.get("data", {}).get(flag, {})
                for bucket in ["0", "1"]:
                    summary = flag_data.get(bucket, {}).get("summary", {})
                    v = summary.get("image", {}).get("custom_pck0.1", {}).get("all",
                        float("nan"))
                    data_rows.append({"backbone": bb, "bucket": int(bucket), "pck@0.1": v})

            flag_df = pd.DataFrame(data_rows)
            bbs     = [b for b in backbones_present if b in flag_df["backbone"].values]
            x       = np.arange(len(bbs))
            w       = 0.35

            for offset, bucket, alpha_v, lbl in [
                (-w / 2, 0, 0.95, "Assente (0)"),
                ( w / 2, 1, 0.45, "Presente (1)"),
            ]:
                sub = flag_df[flag_df["bucket"] == bucket]
                vals = [sub[sub["backbone"] == b]["pck@0.1"].values[0]
                        if not sub[sub["backbone"] == b].empty else float("nan")
                        for b in bbs]
                bars = ax.bar(x + offset, vals, w,
                              color=[BB_COLOR.get(b, "gray") for b in bbs],
                              alpha=alpha_v, label=lbl, edgecolor="white")
                for bar, v in zip(bars, vals):
                    if not np.isnan(v):
                        ax.text(bar.get_x() + bar.get_width() / 2,
                                bar.get_height() + 0.002,
                                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

            ax.set_xticks(x)
            ax.set_xticklabels([BB_LABEL.get(b, b) for b in bbs],
                               fontsize=8.5, rotation=15)
            ax.set_title(FLAG_LABELS[flag], fontweight="bold")
            ax.set_ylabel("PCK@0.1")
            ax.legend(fontsize=8)

        fig.suptitle(
            "PCK@0.1 per flag di difficoltà — miglior modello per backbone",
            fontsize=13, fontweight="bold", y=1.02
        )
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "difficulty_analysis.png", bbox_inches="tight")
        plt.show()
        print("Salvato: figures/difficulty_analysis.png")

## 10 · Riepilogo — Migliori modelli

In [ ]:
print("=" * 72)
print("  RIEPILOGO — MIGLIORE CONFIGURAZIONE PER BACKBONE × METRICA")
print("=" * 72)

summary_rows = []
for bb in backbones_present:
    sub = df[df["backbone"] == bb]
    if sub.empty:
        continue
    for metric in ["pck@0.05", "pck@0.1", "pck@0.2"]:
        best = sub.loc[sub[metric].idxmax()]
        summary_rows.append({
            "Backbone": BB_LABEL[bb],
            "Metrica":  metric.upper(),
            "Valore":   round(float(best[metric]), 5),
            "Run":      best["name"],
        })

summary_df = pd.DataFrame(summary_rows)
display(
    summary_df.style
    .format({"Valore": "{:.5f}"})
    .background_gradient(subset=["Valore"], cmap="Greens")
    .set_caption("Migliore configurazione per backbone × metrica")
)

best_overall = df.loc[df["pck@0.1"].idxmax()]
print(f"\nMiglior risultato assoluto (PCK@0.1):")
print(f"  Run     : {best_overall['name']}")
print(f"  PCK@0.05: {best_overall['pck@0.05']:.4f}")
print(f"  PCK@0.10: {best_overall['pck@0.1']:.4f}")
print(f"  PCK@0.20: {best_overall['pck@0.2']:.4f}")

summary_df.to_csv(FIGURES_DIR / "summary_best_models.csv", index=False)
print("\nSalvato: figures/summary_best_models.csv")

## 11 · Esportazione file di output

In [ ]:
# Tabella completa ordinata come CSV pronto per report
df_export = df_sorted[["name", "backbone", "stage", "blocks", "wsa",
                        "pck@0.05", "pck@0.1", "pck@0.2"]].copy()
df_export["backbone"] = df_export["backbone"].map(lambda b: BB_LABEL.get(b, b))
df_export["stage"]    = df_export["stage"].map(
    {"baseline": "Baseline", "finetune": "Fine-tuning", "lora": "LoRA"})
df_export.rename(columns={
    "pck@0.05": "PCK@0.05", "pck@0.1": "PCK@0.10", "pck@0.2": "PCK@0.20",
    "blocks": "last_blocks", "wsa": "WSA",
}, inplace=True)
df_export.to_csv(FIGURES_DIR / "all_results_sorted.csv", index=False)

print("\nFile generati in ./figures/:")
for f in sorted(FIGURES_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s}  {size:6.1f} KB")